# Training API

Fine-tune forecasting models on your Lightning Rod datasets. This notebook walks through the full training workflow: generating a dataset, estimating cost, creating a training job, and monitoring progress.

The training API supports LoRA fine-tuning with configurable base models, training steps, batch size, and rank.

## Install the SDK

In [1]:
%pip install lightningrod-ai python-dotenv

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [1]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Prepare the dataset

Training requires a dataset ID from a pipeline run. Run one of the other notebooks first to generate a dataset - each one prints the **Dataset ID** after `transforms.run()` — copy it into the cell below.

In [2]:
from lightningrod.training import prepare_for_training
from lightningrod.training.samples import BinaryAnswerType

default_dataset_id = "a2119549-25e6-4deb-87b2-8164949cdb61" # paste it here, or set it as an environment variable
dataset_id = config.get_config_value("DATASET_ID", default_dataset_id)

dataset = lr.datasets.get(dataset_id)
dataset.download()

KeyboardInterrupt: 

In [7]:
import pandas as pd

train, test = prepare_for_training(
    samples=dataset.samples(),
    answer_type=BinaryAnswerType(),
    test_size=0.2,
    split_strategy="random",
    deduplicate_key_fn=lambda sample: (sample.question.question_text, sample.seed.seed_text, sample.label.resolution_date),
    verbose=True,
)

display(pd.DataFrame(train).head())
display(pd.DataFrame(test).head())

[prepare_for_training] Starting with 2000 samples
[filter] 2000 remain (0 dropped)
[dedup] Removed 2 duplicates (2000 → 1998). Top colliding keys:
  ('Will this Hacker News post receive 10 or more upvotes?', Title: Using Nsnotifyd with a PowerDNS Secondary

Content: 

[Computed labels - use for answers]: Upvotes: 2Comments: 0): 2 samples → 1
  ('Will this post receive 5 or more comments?', Title: Using Nsnotifyd with a PowerDNS Secondary

Content: 

[Computed labels - use for answers]: Upvotes: 2Comments: 0): 2 samples → 1
[split] Random split (test_size=0.2): 1598 train, 400 test


,sample_id,prompt,correct_answer,answer_type,reward_function_type,answer_parser_type
0,88d3555b-3b17-4cf6-9f7d-297e7cfe43df,"[{'role': 'user', 'content': 'QUESTION: Will t...",0,binary,binary_log_score,binary
1,9d020f2d-0282-4d86-a883-30cdab768911,"[{'role': 'user', 'content': 'QUESTION: Will t...",0,binary,binary_log_score,binary
2,6daf2382-f900-489e-97d1-ad322103433a,"[{'role': 'user', 'content': 'QUESTION: Will t...",0,binary,binary_log_score,binary
3,e1a01de5-53b2-425a-869e-d270764bbefe,"[{'role': 'user', 'content': 'QUESTION: Will t...",0,binary,binary_log_score,binary
4,404ea524-e97a-40e4-9c36-81d9953dee80,"[{'role': 'user', 'content': 'QUESTION: Will t...",0,binary,binary_log_score,binary


,sample_id,prompt,correct_answer,answer_type,reward_function_type,answer_parser_type
0,bb263177-907d-4b79-87fe-4d41f41dcebd,"[{'role': 'user', 'content': 'QUESTION: Will t...",0,binary,binary_log_score,binary
1,5494bbd1-9929-4731-809b-9f955bdbdca9,"[{'role': 'user', 'content': 'QUESTION: Will t...",0,binary,binary_log_score,binary
2,da973b07-a2a7-4996-807a-48a9b1096a44,"[{'role': 'user', 'content': 'QUESTION: Will t...",0,binary,binary_log_score,binary
3,ffa63c8d-8518-476d-b746-672c79add396,"[{'role': 'user', 'content': 'QUESTION: Will t...",0,binary,binary_log_score,binary
4,2c18cd01-fff4-4aae-b31f-2500a14bc39d,"[{'role': 'user', 'content': 'QUESTION: Will t...",0,binary,binary_log_score,binary


In [8]:
%pip install datasets -q

from datasets import Dataset, DatasetDict
from lightningrod.utils import config

dataset = DatasetDict({
    "train": Dataset.from_list(train),
    "test": Dataset.from_list(test),
})
print(f"Train: {len(dataset['train'])} rows, Test: {len(dataset['test'])} rows")
print("Columns:", dataset["train"].column_names[:8], "...")

DATASET_PATH = f"{config.get_config_value('HF_USERNAME')}/training-demo"
dataset.push_to_hub(DATASET_PATH, token=config.get_config_value("HF_ACCESS_TOKEN"))


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Train: 1598 rows, Test: 400 rows
Columns: ['sample_id', 'prompt', 'correct_answer', 'answer_type', 'reward_function_type', 'answer_parser_type'] ...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/bart/training-demo/commit/606ac55930353bd397cee4aa781a777a6382c90a', commit_message='Upload dataset', commit_description='', oid='606ac55930353bd397cee4aa781a777a6382c90a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/bart/training-demo', endpoint='https://huggingface.co', repo_type='dataset', repo_id='bart/training-demo'), pr_revision=None, pr_num=None)

## Estimate training cost

Before starting a job, use `estimate_cost` to see the expected cost and token usage.

In [10]:
from lightningrod.training import TrainingConfig

config = TrainingConfig(
    dataset_hf_repo=DATASET_PATH,
    base_model="Qwen/Qwen3-4B-Instruct-2507",
    training_steps=10,
)

cost_estimate = lr.training.estimate_cost(config)
print(f"Estimated cost: ${cost_estimate.total_cost_dollars:.2f}")
print(f"Effective steps: {cost_estimate.effective_steps}")
print(f"Train tokens: {cost_estimate.train_tokens:,}")
print(f"Notes: {cost_estimate.notes}")

Estimated cost: $4.53
Effective steps: 10
Train tokens: 10,082,097
Notes: Estimate uses 0.5× max_response_length for output; actual may vary


## Start training

`run` creates a job and polls until completion with a live progress display. Use this when you want to wait for the job to finish in the notebook. Skip this if you used `create` above and prefer to poll manually.

In [11]:
job = lr.training.run(config, name="Forecasting fine-tune")
print(f"Job {job.id} completed with status: {job.status}")
print(f"Trained model ID: {job.model_id}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Training COMPLETED                                                                                          │
│                                                                                                                 │
│    Job: Forecasting fine-tune                                                                                   │
│                                                                                                                 │
│    Reward: latest -0.8394  avg -1.1605  (10 steps)  (higher is better)                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Job cbbc93d1-35f1-4deb-90a1-2c4469242725 completed with status: STARTING
Trained model ID: None


## List and get jobs

List all training jobs or fetch a specific job by ID.

In [6]:
import pandas as pd

jobs_response = lr.training.list(limit=5)

latest_model_id = jobs_response.jobs[-1].model_id

df = pd.DataFrame([
    {
        "Job ID": j.id,
        "Status": j.status,
        "Base Model": getattr(j.config, "base_model", None),
        "Trained Model ID": j.model_id,
    }
    for j in jobs_response.jobs
])

df

,Job ID,Status,Base Model,Trained Model ID
0,cbbc93d1-35f1-4deb-90a1-2c4469242725,COMPLETED,Qwen/Qwen3-4B-Instruct-2507,checkpoint:cbbc93d1-35f1-4deb-90a1-2c4469242725


## Inference with your trained model

Once training completes, use `job.model_id` with the OpenAI-compatible API. We also have a pre-trained foresight-v3 model for forecasting — see [08_foresight_model.ipynb](08_foresight_model.ipynb).

In [4]:
%pip install openai
from IPython.display import clear_output
clear_output()

from openai import OpenAI
from lightningrod.utils import config

base_url = config.get_config_value("LIGHTNINGROD_BASE_URL", "https://api.lightningrod.ai/api/public/v1")
client = OpenAI(api_key=api_key, base_url=f"{base_url}/openai")

In [ ]:
response = client.chat.completions.create(
    model=latest_model_id,
    messages=[
        {"role": "system", "content": "Answer as a probability between 0 and 1 between <answer></answer> tags."},
        {"role": "user", "content": "Will the Fed cut rates by 25bp in March 2026?"}
    ]
)
print(response.choices[0].message.content)

<answer>0.3</answer>


## Run evals on trained model

Run test evals on your trained model against a test dataset. The eval job runs the model on the dataset and reports metrics. Use the same dataset for a quick check, or a separate test split for production.

In [12]:
DATASET_PATH = "bart/training-demo"

eval_job = lr.evals.run(
    model_id=latest_model_id,
    dataset_hf_repo=DATASET_PATH,
)
print(f"Eval {eval_job.id} completed with status: {eval_job.status}")

KeyboardInterrupt: 

In [11]:
import pandas as pd

evals_response = lr.evals.list(limit=5)
pd.DataFrame([
    {
        "Eval ID": e.id,
        "Status": e.status,
        "Error": e.error_message,
        "Test Dataset": e.test_dataset_id,
        "Metrics": dict(e.metrics.additional_properties) if hasattr(e.metrics, "additional_properties") else None,
    }
    for e in evals_response.jobs
])

,Eval ID,Status,Error,Test Dataset,Metrics
0,e0034173-a6dd-4984-8710-1ce59e30743b,FAILED,Can't reach database server at `aws-0-us-east-...,<lightningrod._generated.types.Unset object at...,None
1,4a2b1fb8-9e3c-4bdd-86fe-d5d850d27a82,COMPLETED,None,<lightningrod._generated.types.Unset object at...,"{'base': {'ece': 0.7672499999999968, 'n_valid'..."
2,c9da64dd-3826-44fd-9f68-17920f4feeb9,COMPLETED,None,<lightningrod._generated.types.Unset object at...,"{'base': {'ece': 0.781211711711727, 'n_valid':..."


> Note: the trained model checkpoint will only be available for the period of 7 days. If you wish to host this model long-term, reach out to us at support@lightningrod.ai.